# Random Forest — Official Baseline (RF 02)

This notebook documents the **independent, locked Random Forest baseline** for phishing email classification. It uses the cleaned 53,577-email dataset, the stratified 80/20 split, and the 10,000-feature TF-IDF representation from `rf_01_dataset_preparation.ipynb`. **Class 0 = legitimate; class 1 = phishing.**

**Execution order:** Run `rf_01_dataset_preparation.ipynb` first in the same Python/Colab session. This notebook does **not** download the dataset, clean text, or refit TF-IDF. It trains the baseline Random Forest once, evaluates the unchanged test set, and records the baseline artifacts and figures.

The official results below were reported in the original RF experiment. The code verifies them when executed; this GitHub copy does not claim to contain newly executed outputs.

**Restart guidance:** RF 01 variables exist only in the session where RF 01 ran. For a fresh runtime, either rerun RF 01 before training or use the checkpoint recovery section below to load the previously saved official baseline **without retraining**. Do not run the training cell after recovery if you intend to preserve the locked checkpoint.


## 1. Verify inputs from RF 01

In [ ]:
import hashlib
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve)

required = ["X_train_tfidf", "X_test_tfidf", "y_train", "y_test", "tfidf_vectorizer"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run rf_01_dataset_preparation.ipynb in the same session first. "
        f"Missing: {missing}"
    )

assert X_train_tfidf.shape == (42861, 10000)
assert X_test_tfidf.shape == (10716, 10000)
assert len(y_train) == 42861 and len(y_test) == 10716
assert int(np.sum(np.asarray(y_test) == 0)) == 4748
assert int(np.sum(np.asarray(y_test) == 1)) == 5968
print("Locked dataset split and TF-IDF dimensions verified.")

## 2. Train the official Random Forest baseline

The original baseline uses **100 trees**, `random_state=42`, and `n_jobs=-1`. No class weighting, pruning, threshold adjustment, synthetic training examples, or SHAP intervention is applied.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_tfidf, y_train)

baseline_pred_rf = rf_model.predict(X_test_tfidf)
baseline_prob_rf = rf_model.predict_proba(X_test_tfidf)[:, 1]
y_true_rf = np.asarray(y_test).astype(int).reshape(-1)

assert len(rf_model.estimators_) == 100
assert baseline_pred_rf.shape == (10716,)
assert baseline_prob_rf.shape == (10716,)
print("Official baseline trained and predictions generated.")

## 3. Evaluate and verify official baseline results

With **phishing = positive class (1)**, the standard scikit-learn confusion matrix order is `[[TN, FP], [FN, TP]]`. This means 4,686 correctly classified legitimate emails, 62 false positives, 15 false negatives, and 5,953 correctly classified phishing emails.

In [ ]:
baseline_cm_rf = confusion_matrix(y_true_rf, baseline_pred_rf, labels=[0, 1])
tn_rf, fp_rf, fn_rf, tp_rf = baseline_cm_rf.ravel()

baseline_metrics_rf = {
    "accuracy": accuracy_score(y_true_rf, baseline_pred_rf),
    "precision": precision_score(y_true_rf, baseline_pred_rf, pos_label=1),
    "recall": recall_score(y_true_rf, baseline_pred_rf, pos_label=1),
    "f1": f1_score(y_true_rf, baseline_pred_rf, pos_label=1),
    "roc_auc": roc_auc_score(y_true_rf, baseline_prob_rf),
    "fpr": fp_rf / (fp_rf + tn_rf),
    "fnr": fn_rf / (fn_rf + tp_rf),
}

assert baseline_cm_rf.tolist() == [[4686, 62], [15, 5953]], baseline_cm_rf
expected_metrics_rf = {
    "accuracy": 0.992814483016,
    "precision": 0.989692435578,
    "recall": 0.997486595174,
    "f1": 0.993574230159,
    "roc_auc": 0.999113532493,
}
for metric, expected in expected_metrics_rf.items():
    assert np.isclose(baseline_metrics_rf[metric], expected, atol=1e-9), metric

print("Confusion matrix [[TN, FP], [FN, TP]]:\n", baseline_cm_rf)
for metric, value in baseline_metrics_rf.items():
    print(f"{metric}: {value:.12f}")
print("Official baseline verification: PASSED")

### Recorded official baseline (original experiment)

| Measure | Value |
|---|---:|
| Correct legitimate (TN, standard convention) | 4,686 |
| False positives (FP) | 62 |
| False negatives (FN) | 15 |
| Correct phishing (TP, standard convention) | 5,953 |
| Accuracy | 0.99281448 |
| Precision | 0.98969244 |
| Recall | 0.99748660 |
| F1-score | 0.99357423 |
| ROC-AUC | 0.99911353 |
| False-positive rate | 0.01305813 |
| False-negative rate | 0.00251340 |

**Predefined RF improvement condition:** Reduce FP below **62** while keeping FN at **15 or fewer**. Later experiments are documented separately; they must not change this official baseline.

## 4. Identify baseline errors

In [ ]:
false_positive_positions_rf = np.flatnonzero((y_true_rf == 0) & (baseline_pred_rf == 1))
false_negative_positions_rf = np.flatnonzero((y_true_rf == 1) & (baseline_pred_rf == 0))
assert len(false_positive_positions_rf) == 62
assert len(false_negative_positions_rf) == 15
print("False positives:", len(false_positive_positions_rf))
print("False negatives:", len(false_negative_positions_rf))

## 5. Save the locked baseline artifacts (original Step 5B)

This cell exports the trained baseline and its unchanged predictions to the original Google Drive project directory. **Mount Google Drive first** in Colab using `from google.colab import drive; drive.mount('/content/drive')`. If Drive is not mounted, the cell stops with a clear message rather than claiming the checkpoint was saved. The original experiment used a restart-safe baseline lock; the next section verifies a saved checkpoint and can recover it in a fresh session without retraining. Large binary files are intended for Drive, not the GitHub repository.

In [ ]:
# In Colab, mount Drive separately before running this cell:
# from google.colab import drive
# drive.mount('/content/drive')

baseline_dir_rf = Path(
    "/content/drive/MyDrive/Random_Forest_SHAP_Guided_Phishing_Detection/"
    "Official_Random_Forest_Baseline"
)

# Avoid silently saving somewhere else if Google Drive is not mounted.
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Mount Google Drive before saving the baseline checkpoint.")
else:
    baseline_dir_rf.mkdir(parents=True, exist_ok=True)
    joblib.dump(rf_model, baseline_dir_rf / "rf_baseline_model.joblib")
    joblib.dump(tfidf_vectorizer, baseline_dir_rf / "tfidf_vectorizer.joblib")
    sparse.save_npz(baseline_dir_rf / "X_train_tfidf.npz", X_train_tfidf)
    sparse.save_npz(baseline_dir_rf / "X_test_tfidf.npz", X_test_tfidf)
    np.save(baseline_dir_rf / "y_train.npy", np.asarray(y_train))
    np.save(baseline_dir_rf / "y_test.npy", y_true_rf)
    np.save(baseline_dir_rf / "baseline_predictions.npy", baseline_pred_rf)
    np.save(baseline_dir_rf / "baseline_probabilities.npy", baseline_prob_rf)
    prediction_sha256_rf = hashlib.sha256(
        np.ascontiguousarray(baseline_pred_rf).tobytes()
    ).hexdigest()
    manifest_rf = {
        "model": "RandomForestClassifier",
        "n_estimators": 100,
        "random_state": 42,
        "n_jobs": -1,
        "train_rows": 42861,
        "test_rows": 10716,
        "tfidf_features": 10000,
        "confusion_matrix_sklearn": baseline_cm_rf.tolist(),
        "metrics": baseline_metrics_rf,
        "predictions_sha256": prediction_sha256_rf,
    }
    with open(baseline_dir_rf / "baseline_manifest.json", "w") as f:
        json.dump(manifest_rf, f, indent=2)
    print("Baseline checkpoint saved to:", baseline_dir_rf)
    print("Predictions SHA256:", prediction_sha256_rf)
    assert prediction_sha256_rf == "2c7f695b0dfae480ae17a5e59bb661b4e76606d45dc7e30bf2e8f0c74230cc2e", "Saved baseline predictions differ from the original locked baseline."

## 5B. Restart-safe baseline recovery and verification (no retraining)

**Alternative to Sections 1–5 in a fresh runtime:** Mount the same Google Drive and run the following recovery cell. It loads the previously exported model, vectorizer, sparse TF-IDF matrices, labels, predictions, probabilities, and manifest. It verifies their shapes, the original baseline confusion matrix, metrics, and prediction SHA-256. **It does not refit TF-IDF or retrain the model.**

Run this cell only after a successful checkpoint export exists. The filename convention here matches the checkpoint export in this GitHub notebook; if your original Drive checkpoint uses different filenames, use those actual filenames rather than silently substituting another model.

In [ ]:
# Run in a fresh Colab session only AFTER mounting the original Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

import hashlib
import json
from pathlib import Path
import joblib
import numpy as np
from scipy import sparse
from sklearn.metrics import confusion_matrix, roc_auc_score

baseline_dir_rf = Path(
    "/content/drive/MyDrive/Random_Forest_SHAP_Guided_Phishing_Detection/"
    "Official_Random_Forest_Baseline"
)
needed_rf = [
    "rf_baseline_model.joblib", "tfidf_vectorizer.joblib",
    "X_train_tfidf.npz", "X_test_tfidf.npz", "y_train.npy", "y_test.npy",
    "baseline_predictions.npy", "baseline_probabilities.npy",
    "baseline_manifest.json",
]
missing_rf = [name for name in needed_rf if not (baseline_dir_rf / name).is_file()]
if missing_rf:
    raise FileNotFoundError(
        f"Official RF baseline checkpoint is incomplete at {baseline_dir_rf}: {missing_rf}"
    )

rf_model = joblib.load(baseline_dir_rf / "rf_baseline_model.joblib")
tfidf_vectorizer = joblib.load(baseline_dir_rf / "tfidf_vectorizer.joblib")
X_train_tfidf = sparse.load_npz(baseline_dir_rf / "X_train_tfidf.npz")
X_test_tfidf = sparse.load_npz(baseline_dir_rf / "X_test_tfidf.npz")
y_train = np.load(baseline_dir_rf / "y_train.npy", allow_pickle=False)
y_test = np.load(baseline_dir_rf / "y_test.npy", allow_pickle=False)
baseline_pred_rf = np.load(baseline_dir_rf / "baseline_predictions.npy", allow_pickle=False)
baseline_prob_rf = np.load(baseline_dir_rf / "baseline_probabilities.npy", allow_pickle=False)
with open(baseline_dir_rf / "baseline_manifest.json", encoding="utf-8") as f:
    manifest_rf = json.load(f)

y_true_rf = np.asarray(y_test).astype(int).reshape(-1)
assert X_train_tfidf.shape == (42861, 10000)
assert X_test_tfidf.shape == (10716, 10000)
assert y_train.shape[0] == 42861 and y_true_rf.shape[0] == 10716
assert baseline_pred_rf.shape == (10716,)
assert baseline_prob_rf.shape == (10716,)
assert len(rf_model.estimators_) == 100
assert manifest_rf["n_estimators"] == 100
assert manifest_rf["random_state"] == 42

baseline_cm_rf = confusion_matrix(y_true_rf, baseline_pred_rf, labels=[0, 1])
assert baseline_cm_rf.tolist() == [[4686, 62], [15, 5953]]
assert manifest_rf["confusion_matrix_sklearn"] == baseline_cm_rf.tolist()
assert np.isclose(roc_auc_score(y_true_rf, baseline_prob_rf), 0.999113532493, atol=1e-9)

prediction_sha256_rf = hashlib.sha256(
    np.ascontiguousarray(baseline_pred_rf).tobytes()
).hexdigest()
assert prediction_sha256_rf == manifest_rf["predictions_sha256"]
assert prediction_sha256_rf == (
    "2c7f695b0dfae480ae17a5e59bb661b4e76606d45dc7e30bf2e8f0c74230cc2e"
), "Loaded predictions do not match the recorded official baseline hash."

print("Official RF baseline recovered and verified without retraining.")
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", baseline_cm_rf)
print("Prediction SHA256:", prediction_sha256_rf)

## 6. Baseline confusion matrix figure

The original RF baseline experiment generated a confusion matrix image at 300 DPI.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(baseline_cm_rf, cmap="Blues")
ax.set_xticks([0, 1], labels=["Legitimate (0)", "Phishing (1)"])
ax.set_yticks([0, 1], labels=["Legitimate (0)", "Phishing (1)"])
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title("Random Forest Baseline Confusion Matrix")
for (row, col), value in np.ndenumerate(baseline_cm_rf):
    ax.text(col, row, str(value), ha="center", va="center",
            color="white" if value > baseline_cm_rf.max() / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig("Random_Forest_Baseline_Confusion_Matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Correctly classified emails and error-count bar charts

These reproduce the two original baseline figure types, using the verified baseline counts.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(["Correct legitimate", "Correct phishing"], [tn_rf, tp_rf])
ax.bar_label(bars, padding=3)
ax.set_ylabel("Number of emails")
ax.set_title("Random Forest Baseline: Correct Classifications")
ax.set_ylim(0, 6500)
fig.tight_layout()
fig.savefig("Random_Forest_Baseline_TP_TN_Histogram.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(["False positives", "False negatives"], [fp_rf, fn_rf])
ax.bar_label(bars, padding=3)
ax.set_ylabel("Number of emails")
ax.set_title("Random Forest Baseline: FP and FN")
ax.set_ylim(0, 75)
fig.tight_layout()
fig.savefig("Random_Forest_Baseline_FP_FN_Histogram.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Official baseline ROC curve

In [ ]:
baseline_fpr_curve_rf, baseline_tpr_curve_rf, _ = roc_curve(
    y_true_rf, baseline_prob_rf
)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(baseline_fpr_curve_rf, baseline_tpr_curve_rf,
        label=f"Random Forest (AUC = {baseline_metrics_rf['roc_auc']:.6f})", linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="Random Forest Baseline ROC Curve", xlim=(0, 1), ylim=(0, 1.05))
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("Random_Forest_Baseline_ROC_AUC.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Baseline lock and next stage

The **official baseline is 62 FP and 15 FN**, with the confusion matrix and metrics shown above. Subsequent SHAP, feature-dropout, counterfactual, synthetic-email, and final-model experiments are evaluated independently against this locked baseline. This notebook does not contain the later final model.

**Reproducibility note:** The exact results depend on the same cleaned data, scikit-learn version, TF-IDF representation, and training split used in the original experiment. The figures are generated when the notebook runs; they are not embedded as pre-existing execution outputs in this GitHub copy.

**Checkpoint limitation:** The recovery cell validates the prediction checksum and key recorded metrics. It does not claim to reproduce the full original Step 5B artifact-by-artifact SHA-256 manifest or verify original model bytes.
